# Results analysis

Load prediction parquet output, compute metrics via `aev_plig.results`, and render Plotly figures.


In [ ]:
from pathlib import Path

import plotly.express as px
import plotly.graph_objects as go
import polars as pl

from aev_plig import results


In [ ]:
PREDICTIONS_PATH = "output/predictions/example_predictions.parquet"
TRUTH_COL = "pK"
PRED_COL = "preds"
TARGET_COL = "pdb_file"  # adjust if you want a different grouping
MIN_SAMPLES = 1
FIG_DIR = Path("output/figures")
SAVE_HTML = False


In [ ]:
df = results.load_predictions(PREDICTIONS_PATH)
df.shape


In [ ]:
df.select(["unique_id", "graph_id", TRUTH_COL, PRED_COL]).head()


In [ ]:
overall = results.overall_metrics(df, truth_col=TRUTH_COL, pred_col=PRED_COL)
overall


In [ ]:
plot_df = df.select([TRUTH_COL, PRED_COL, "unique_id", "graph_id"]).drop_nulls()
fig_scatter = px.scatter(
    plot_df.to_pandas(),
    x=TRUTH_COL,
    y=PRED_COL,
    hover_data=["unique_id", "graph_id"],
    title="Predicted vs True",
)
x_min = float(plot_df[TRUTH_COL].min())
x_max = float(plot_df[TRUTH_COL].max())
fig_scatter.add_trace(go.Scatter(x=[x_min, x_max], y=[x_min, x_max], mode="lines", name="y=x"))
fig_scatter.show()


In [ ]:
residual_df = df.select([TRUTH_COL, PRED_COL]).drop_nulls().with_columns((pl.col(PRED_COL) - pl.col(TRUTH_COL)).alias("residual"))
fig_resid = px.histogram(residual_df.to_pandas(), x="residual", nbins=30, title="Residual distribution")
fig_resid.show()


In [ ]:
by_target = results.per_target_metrics(
    df,
    target_col=TARGET_COL,
    truth_col=TRUTH_COL,
    pred_col=PRED_COL,
    min_samples=MIN_SAMPLES,
)
by_target.sort("rmse")


In [ ]:
if by_target.height > 0:
    fig_target = px.bar(by_target.to_pandas(), x="target", y="rmse", title="Per-target RMSE")
    fig_target.update_layout(xaxis_tickangle=45)
    fig_target.show()
else:
    print("No groups matched min_samples")


In [ ]:
if SAVE_HTML:
    FIG_DIR.mkdir(parents=True, exist_ok=True)
    fig_scatter.write_html(FIG_DIR / "pred_vs_true.html")
    fig_resid.write_html(FIG_DIR / "residual_hist.html")
    if by_target.height > 0:
        fig_target.write_html(FIG_DIR / "per_target_rmse.html")
    print(f"Saved figures to {FIG_DIR}")
